In [0]:
# Install MLflow
%pip install mlflow
%pip install prophet
dbutils.library.restartPython()

In [0]:
import mlflow
import prophet
import pandas as pd
import numpy as np
from prophet import Prophet
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
mlflow.set_experiment(experiment_id="2490562988127812")

In [0]:
# ---------------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------------
FEATURE_TABLE     = "h2b_dbx_salesorder.featuredatasets.material_lvl"
WEATHER_TABLE     = "h2b_bdc_weather.weather.weather"
RESULT_CATALOG    = "h2b_dbx_salesorder"
RESULT_SCHEMA     = "forecasts"
RESULT_TABLE      = "forecast_salesorderquantity_material"


TRAINING_END      = "2025-12-31"
FORECAST_HORIZON  = 26   # weeks

# CM-MLFL-KM-VXX excluded — phased out 2025-06-01, no forecast needed
FORECAST_MATERIALS = ["TG11", "TG12", "FPP", "RTE", "CM-FL-V00"]

# DE stations carry the heat-wave signal; Tier A (91% revenue) is German market
WEATHER_COUNTRIES  = ["DE"]

In [0]:
# ---------------------------------------------------------------------------
# Load & prepare training data
# ---------------------------------------------------------------------------
history_pdf = (
    spark.table(FEATURE_TABLE)
    .withColumn("SalesOrderDate", F.to_timestamp("SalesOrderDate"))
    .filter(F.col("SalesOrderDate") <= TRAINING_END)
    .filter(F.col("Material").isin(FORECAST_MATERIALS))
    .select("SalesOrderDate", "Material", "MaterialGroup", "Sum_OrderQuantity")
    .toPandas()
)
history_pdf["SalesOrderDate"] = pd.to_datetime(history_pdf["SalesOrderDate"])

print(f"Training rows : {len(history_pdf):,}")
print(f"Date range    : {history_pdf['SalesOrderDate'].min().date()} → {history_pdf['SalesOrderDate'].max().date()}")
print(f"Materials     : {sorted(history_pdf['Material'].unique())}")
display(history_pdf.groupby("Material")["Sum_OrderQuantity"].describe().round(0))

In [0]:
# ---------------------------------------------------------------------------
# Prepare weekly weather regressors
# Monthly DE weather → interpolated to weekly for Prophet regressors
# ---------------------------------------------------------------------------
weather_pdf = spark.table(WEATHER_TABLE).toPandas()

de_monthly = (
    weather_pdf[weather_pdf["country"].isin(WEATHER_COUNTRIES)]
    .groupby(["year", "month"], as_index=False)
    .agg(avg_temp_c=("avg_temp_c", "mean"), temp_anomaly_c=("temp_anomaly_c", "mean"))
)
de_monthly["month_start"] = pd.to_datetime(
    de_monthly["year"].astype(str) + "-" +
    de_monthly["month"].astype(str).str.zfill(2) + "-01"
)

# Build a weekly index spanning the full 42-month window (training + forecast)
monthly_idx = de_monthly.set_index("month_start")[["avg_temp_c", "temp_anomaly_c"]].sort_index()
full_range   = pd.date_range(monthly_idx.index.min(),
                              monthly_idx.index.max() + pd.offsets.MonthEnd(1),
                              freq="W")
weather_weekly = (
    monthly_idx
    .reindex(monthly_idx.index.union(full_range))
    .interpolate(method="time")
    .reindex(full_range)
    .reset_index()
    .rename(columns={"index": "ds"})
)
weather_weekly["ds"] = pd.to_datetime(weather_weekly["ds"])

print(f"Weather weekly: {len(weather_weekly)} rows  "
      f"{weather_weekly['ds'].min().date()} → {weather_weekly['ds'].max().date()}")

In [0]:
# ---------------------------------------------------------------------------
# Prophet training helper — one model per material
# ---------------------------------------------------------------------------

def _merge_weather(df, ds_col, weather_weekly):
    """Left-join weekly weather onto df; ffill/bfill any edge gaps."""
    out = df.merge(
        weather_weekly[["ds", "avg_temp_c", "temp_anomaly_c"]],
        left_on=ds_col, right_on="ds", how="left"
    ).drop(columns=["ds"], errors="ignore")
    out[["avg_temp_c", "temp_anomaly_c"]] = (
        out[["avg_temp_c", "temp_anomaly_c"]].ffill().bfill()
    )
    return out


def train_prophet(material, mat_df, weather_weekly, forecast_horizon):
    """
    Train a Prophet model for one material.
    Logs params, metrics, and model artifact to MLflow.
    Returns a DataFrame of forecast rows for the future window only.
    """
    mat_group = mat_df["MaterialGroup"].iloc[0]

    train = (
        mat_df[["SalesOrderDate", "Sum_OrderQuantity"]]
        .rename(columns={"SalesOrderDate": "ds", "Sum_OrderQuantity": "y"})
        .sort_values("ds")
        .reset_index(drop=True)
    )
    train = _merge_weather(train, "ds", weather_weekly)

    with mlflow.start_run(run_name=f"prophet_{material}"):
        mlflow.log_params({
            "material":               material,
            "material_group":         mat_group,
            "training_rows":          len(train),
            "training_end":           str(train["ds"].max().date()),
            "forecast_horizon_weeks": forecast_horizon,
            "seasonality_mode":       "multiplicative",
            "weather_regressors":     "avg_temp_c, temp_anomaly_c",
        })

        m = Prophet(
            yearly_seasonality  = True,
            weekly_seasonality  = False,   # data is already weekly
            daily_seasonality   = False,
            seasonality_mode    = "multiplicative",
            interval_width      = 0.80,
        )
        m.add_regressor("avg_temp_c")
        m.add_regressor("temp_anomaly_c")
        m.fit(train)

        # Build future dataframe with weather regressors
        future = m.make_future_dataframe(periods=forecast_horizon, freq="W")
        future = _merge_weather(future, "ds", weather_weekly)
        forecast = m.predict(future)

        # Training-window fit metrics
        fit = forecast[forecast["ds"].isin(train["ds"])][["ds", "yhat"]].merge(
            train[["ds", "y"]], on="ds"
        )
        mae  = float(np.mean(np.abs(fit["y"] - fit["yhat"])))
        mape = float(np.mean(np.abs((fit["y"] - fit["yhat"]) / fit["y"].clip(lower=1))) * 100)
        mlflow.log_metrics({"train_mae": round(mae, 2), "train_mape_pct": round(mape, 2)})
        mlflow.prophet.log_model(m, artifact_path=f"prophet_{material}")

        print(f"  {material:<20s}  train_rows={len(train):>3}  MAE={mae:,.0f}  MAPE={mape:.1f}%")

        # Return forecast-window rows only
        cutoff = train["ds"].max()
        out = forecast[forecast["ds"] > cutoff][
            ["ds", "yhat", "yhat_lower", "yhat_upper"]
        ].copy()
        out["Material"]      = material
        out["MaterialGroup"] = mat_group
        return out

In [0]:
# ---------------------------------------------------------------------------
# Train one Prophet model per material
# ---------------------------------------------------------------------------
all_forecasts = []
print("Training Prophet models...\n")

for material in FORECAST_MATERIALS:
    mat_df = history_pdf[history_pdf["Material"] == material].copy()
    if len(mat_df) < 10:
        print(f"  {material}: skipping — only {len(mat_df)} rows")
        continue
    fc = train_prophet(material, mat_df, weather_weekly, FORECAST_HORIZON)
    all_forecasts.append(fc)

forecast_pdf = pd.concat(all_forecasts, ignore_index=True)
print(f"\nForecast rows: {len(forecast_pdf):,}  "
      f"({forecast_pdf['ds'].min().date()} → {forecast_pdf['ds'].max().date()})")

In [0]:
# ---------------------------------------------------------------------------
# Join actuals from feature table (test window) and save results
# ---------------------------------------------------------------------------

# Actuals for the forecast window — present in material_lvl if table covers 42 months
actuals_spark = (
    spark.table(FEATURE_TABLE)
    .withColumn("SalesOrderDate", F.to_timestamp("SalesOrderDate"))
    .filter(F.col("SalesOrderDate") > TRAINING_END)
    .filter(F.col("Material").isin(FORECAST_MATERIALS))
    .select(
        F.col("SalesOrderDate"),
        F.col("Material"),
        F.col("MaterialGroup"),
        F.col("Sum_OrderQuantity").alias("actual_qty"),
    )
)

forecast_spark = spark.createDataFrame(
    forecast_pdf.rename(columns={
        "ds":          "SalesOrderDate",
        "yhat":        "predicted_qty",
        "yhat_lower":  "predicted_qty_lower",
        "yhat_upper":  "predicted_qty_upper",
    })
)

results = (
    forecast_spark
    .join(actuals_spark, on=["SalesOrderDate", "Material", "MaterialGroup"], how="left")
    .withColumn(
        "gap_actual_vs_forecast",
        F.col("actual_qty") - F.col("predicted_qty"),
    )
    .withColumn(
        "pct_error",
        F.when(
            F.col("actual_qty").isNotNull(),
            F.round(
                F.abs(F.col("gap_actual_vs_forecast")) / F.col("actual_qty") * 100, 2
            ),
        ),
    )
    .orderBy("Material", "SalesOrderDate")
)

# Persist to results catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {RESULT_CATALOG}.{RESULT_SCHEMA}")
(
    results.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{RESULT_CATALOG}.{RESULT_SCHEMA}.{RESULT_TABLE}")
)

print(f"Saved {results.count():,} rows → {RESULT_CATALOG}.{RESULT_SCHEMA}.{RESULT_TABLE}")
display(results)